# 4 · Exploratory data analysis

| | |
|---|---|
| Lifecycle stage | 4 of 8: *EDA* (understand patterns, test the stage-1 hypotheses) |
| Tools | pandas, Seaborn, Matplotlib, Plotly |
| Code | `src/fraudlens/eda.py`, `src/fraudlens/interactive.py` |
| Data | **training period only** (Jan 2019 to May 2020), so nothing learned here leaks from validation or test |
| Outputs | `reports/figures/01-09_*.png`, `reports/eda_summary.json`, `reports/interactive/eda_explorer.html` |

In [1]:
import os, sys, json, warnings
from pathlib import Path
ROOT = Path(os.environ.get("FRAUDLENS_ROOT") or (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
CODE = Path(os.environ.get("FRAUDLENS_CODE", ROOT)).resolve()
sys.path.insert(0, str(CODE / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from fraudlens import viz
from fraudlens.utils import load_config, read_json, read_table

viz.setup()
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"   # small notebooks: plotly.js from the CDN
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
cfg = load_config(root=ROOT)
P = cfg["paths"]
print(f"Project root: {ROOT.name} | data present: {(P['silver'] / 'transactions.parquet').exists()}")
from fraudlens import eda
from fraudlens import features as F
from fraudlens.datasets import load_analysis_frame
df = load_analysis_frame(cfg, "train")
numeric = F.numeric_features(cfg["features"]["velocity_windows"])
print(f"{len(df):,} training transactions, {int(df['is_fraud'].sum()):,} frauds ({df['is_fraud'].mean():.3%})")

Project root: fraudlens | data present: True


1,185,094 training transactions, 6,837 frauds (0.577%)


## 4.1 Class balance: rare, and rarer in the test period

About one transaction in 170 is fraud during training, and the test period is even more imbalanced.
Precision at a fixed threshold will therefore be lower on test than on validation, for reasons that have
nothing to do with the model.

In [2]:
t, fig = eda.class_balance(cfg)
plt.show()
t

,transactions,frauds,fraud_rate_pct,legit_per_fraud
split,,,,
train,1185094,6837,0.5770,172.3000
val,111581,669,0.6000,165.8000
test,555719,2145,0.3860,258.1000


## 4.2 H1: fraudulent transactions are much larger

In [3]:
t, fig = eda.amount_profile(df)
plt.show()
display(t)
fig = eda.amount_by_category_seaborn(df)
plt.show()

,mean,p5,p25,p50,p75,p95,p99
is_fraud,,,,,,,
legit,67.6700,2.4400,9.6200,47.2800,82.5500,190.0200,486.4100
fraud,530.1400,9.1000,245.2300,387.4100,900.8600,"1,084.5000","1,179.9200"


In [4]:
top5 = df["amount"] >= df["amount"].quantile(0.95)
print(f"The top 5% of amounts contain {df.loc[top5, 'is_fraud'].sum() / df['is_fraud'].sum():.0%} of all fraud "
      f"(fraud rate {df.loc[top5, 'is_fraud'].mean():.2%} vs {df['is_fraud'].mean():.2%} overall).")

The top 5% of amounts contain 76% of all fraud (fraud rate 8.76% vs 0.58% overall).


## 4.3 H4: merchant category matters

In [5]:
t, fig = eda.fraud_by_category(df)
plt.show()
t.head(8)

,transactions,frauds,fraud_amount,fraud_rate_pct,share_of_frauds_pct
category,,,,,
shopping_net,89100,1562,"1,561,680.9900",1.7530,22.8460
misc_net,57693,821,"653,596.1300",1.4230,12.0080
grocery_pos,113103,1599,"498,557.1500",1.4140,23.3870
shopping_pos,106592,768,"672,805.0000",0.7210,11.2330
gas_transport,120364,562,"6,893.9600",0.4670,8.2200
misc_pos,72789,224,"45,574.2400",0.3080,3.2760
grocery_net,41494,125,"1,516.9700",0.3010,1.8280
travel,37070,108,969.2700,0.2910,1.5800


## 4.4 H2: fraud happens at night

In [6]:
tabs, fig = eda.fraud_by_time(df)
plt.show()
night = df["is_night"] == 1
print(f"Night hours (22:00-03:59): {night.mean():.1%} of transactions but {df.loc[night, 'is_fraud'].sum() / df['is_fraud'].sum():.1%} of fraud; "
      f"fraud rate {df.loc[night, 'is_fraud'].mean():.2%} at night vs {df.loc[~night, 'is_fraud'].mean():.2%} by day.")

Night hours (22:00-03:59): 23.5% of transactions but 84.8% of fraud; fraud rate 2.08% at night vs 0.11% by day.


## 4.5 H5: demographics carry little signal

In [7]:
tabs, fig = eda.fraud_by_demographics(df, cfg["eda"]["min_state_txns"])
plt.show()
pd.concat({k: v["fraud_rate_pct"] for k, v in tabs.items() if k != "state"}, axis=0).to_frame("fraud_rate_pct").T

age_band                                           gender       
                    <25  25-34  35-44  45-54  55-64  65-74    75+      F      M
fraud_rate_pct   0.6560 0.4730 0.4200 0.5800 0.7610 0.5800 0.9190 0.5230 0.6420

## 4.6 H3: fraud comes in bursts on a compromised card

In [8]:
tabs, fig = eda.fraud_by_behaviour(df)
plt.show()
cards = eda.card_level_patterns(df)
pd.Series(cards).to_frame("value").T

,cards_total,cards_with_fraud,frauds_per_fraud_card_median,frauds_per_fraud_card_max,fraud_span_hours_median,fraud_span_hours_p90,fraud_amount_per_card_median
value,978.0000,694.0000,10.0000,19.0000,45.2000,47.2000,"5,146.6200"


## 4.7 Which single features separate fraud best?

Single-feature ROC AUC (0.5 = no signal) on a stratified sample, and the Spearman correlation between
features: highly correlated pairs (e.g. `amount` and `log_amount`) are fine for trees but are reduced for
the linear model.

In [9]:
t, fig = eda.feature_signal(df, numeric, cfg["eda"]["sample_rows"])
plt.show()
t.head(12)

,auc,direction,mean_legit,mean_fraud
feature,,,,
amt_sum_24h,0.8597,higher=riskier,264.8215,"1,921.9116"
log_amount,0.8356,higher=riskier,3.5221,5.5654
amount,0.8356,higher=riskier,67.6694,530.1361
amt_to_card_mean,0.8120,higher=riskier,0.9884,6.9364
is_night,0.8100,higher=riskier,0.2316,0.8485
amt_zscore_card,0.8023,higher=riskier,0.0107,4.0889
amt_sum_7d,0.7368,higher=riskier,"1,740.3927","3,328.6186"
amt_sum_1h,0.6691,higher=riskier,12.1700,412.6695
secs_since_last_txn,0.6558,lower=riskier,"32,820.0696","21,349.0464"


In [10]:
pairs, fig = eda.correlations(df, numeric, cfg["eda"]["sample_rows"], cfg["eda"]["corr_threshold"])
plt.show()
pairs.head(8)

,level_0,level_1,spearman_rho
0,amount,log_amount,1.0000
1,txn_count_1h,amt_sum_1h,0.9940
2,log_amount,amt_to_card_mean,0.9720
3,amount,amt_to_card_mean,0.9720
4,amt_to_card_mean,amt_zscore_card,0.9220
5,card_prior_txn_count,card_prior_category_count,0.9190
6,amount,amt_zscore_card,0.8840
7,log_amount,amt_zscore_card,0.8840


## 4.8 Interactive views (Plotly)

The same analysis as interactive charts: fraud rate by category × hour of day, and the amount distribution
by class. The full interactive explorer is `reports/interactive/eda_explorer.html` (open it in a browser).

In [11]:
import plotly.express as px
import plotly.io as pio
pio.templates["fraudlens"] = viz.plotly_template()
pio.templates.default = "fraudlens"

heat = (df.groupby(["category", "hour"], observed=True)["is_fraud"].mean().mul(100).rename("fraud_rate_pct")
          .reset_index().pivot(index="category", columns="hour", values="fraud_rate_pct"))
fig = px.imshow(heat, aspect="auto", color_continuous_scale=viz.BLUE_RAMP,
                labels={"color": "Fraud rate (%)", "x": "Hour of day", "y": ""},
                title="Fraud rate by merchant category and hour (training period)")
fig.show()

In [12]:
rng = np.random.default_rng(0)
legit = df[df["is_fraud"] == 0]
s = pd.concat([df[df["is_fraud"] == 1], legit.iloc[rng.choice(len(legit), min(60_000, len(legit)), replace=False)]])
s = s.assign(Class=s["is_fraud"].map({0: "Legitimate", 1: "Fraud"}), log10_amount=np.log10(s["amount"]))
fig = px.histogram(s, x="log10_amount", color="Class", histnorm="probability density", barmode="overlay", nbins=80,
                   opacity=0.6, color_discrete_map={"Legitimate": viz.LEGIT, "Fraud": viz.FRAUD},
                   title="Amount by class (log10 $): fraud clusters at $200-$1,200")
fig.show()

## 4.9 Drift between periods (a preview of stage 8)

Population Stability Index of each feature between training and the later periods. Running totals such
as `card_prior_txn_count` drift by construction (they only grow) and were excluded from modelling; the
behavioural ratios and windows are stable.

In [13]:
summary = read_json(P["reports"] / "eda_summary.json")
drift = pd.DataFrame(summary["drift_psi"]).T.sort_values("test", ascending=False)
display(drift.head(10))
pd.DataFrame(summary["new_cards"]).T

,val,test,flag_val,flag_test
card_prior_txn_count,5.3707,4.5001,major,major
card_prior_category_count,1.4909,2.1729,major,major
is_new_merchant_for_card,0.3888,0.5642,major,major
txn_count_7d,0.0306,0.0946,stable,stable
day_of_week,0.0562,0.0650,stable,stable
amt_sum_7d,0.0149,0.0569,stable,stable
amt_zscore_card,0.0385,0.0526,stable,stable
txn_count_24h,0.0096,0.0503,stable,stable
amt_sum_24h,0.0066,0.0316,stable,stable
is_weekend,0.0198,0.0256,stable,stable


,cards,cards_not_in_train,transactions_on_new_cards
val,914,5,50
test,924,16,163


## 4.10 What EDA tells the modelling stage

| Hypothesis | Verdict | Consequence for modelling |
|---|---|---|
| H1 large amounts | **Strongly supported**: median fraud ≈ $390 vs ≈ $47 legitimate; most fraud is in the top 5% of amounts | Keep outliers; use amount and amount relative to the card's history |
| H2 night | **Strongly supported**: about 85% of training fraud happens between 22:00 and 03:59, where the fraud rate is ~18× the daytime rate | `hour`, `is_night` |
| H3 bursts | **Supported**: fraud cards see ~10 frauds within ~2 days; 24-hour spend is the single strongest feature | velocity windows 1 h / 24 h / 7 d |
| H4 category | **Supported**: online shopping, misc. online and grocery POS have 2-3× the base rate | `category` (native categorical in trees) |
| H5 demographics | **Weak**: age bands differ modestly, gender barely | keep `age`, `gender` but monitor fairness (stage 6) |

Other findings: fraud is rarer in the test period; a few cards appear only after training (cold start, handled
by the online feature store); feature pairs with |ρ| > 0.85 are kept for trees and handled for the linear model.

**Next:** [05 · Modelling](05_modelling.ipynb)